In [4]:
'''
Updated 24-01-2023

Form groups for every individual targets
'''
import pandas as pd
import numpy as np
from statistics import mode

# Load metadata + spectra
metadata = pd.read_pickle("/home/msp25gd/Downloads/res/meta/metadata.pkl")
# sK = np.load("/home/msp25gd/Downloads/res/spec/sK.npy")
# sH = np.load("/home/msp25gd/Downloads/res/spec/sH.npy")
# fits_list = np.load("/home/msp25gd/Downloads/res/fits_list.npy")
print('metadata + spectra loaded')

### Get Groups according to reduced name only ###
# Group metadata by reduced name (groupby object)
r = metadata.groupby(['Reduced'], as_index=False)

# Assign group number to the grouped reduced name
metadata['Nb Obs'] = np.ones(len(metadata), dtype=int)
agg_func_count = {'Nb Obs': 'sum'}
reduced = r.agg(agg_func_count)

reduced.insert(0, 'Groups', np.arange(len(reduced)))
reduced = reduced.rename(columns={'Groups': 'New Groups'})

### Map back the group number to metadata ###
meta = metadata.copy()  # keep metadata authentic
full = meta.join(reduced.set_index('Reduced')['New Groups'], on='Reduced')
full = full.drop(columns=['Nb Obs'])

# Get all information for all the different groups
full_grouped = full.groupby('New Groups')

all_groups = full_grouped.first()  # df to be updated with new data
if 'Sanitised' in full.columns:
    all_groups['Sanitised'] = full_grouped['Sanitised'].apply(lambda san: mode(san))
if 'Reduced' in full.columns:
    all_groups['Reduced'] = full_grouped['Reduced'].apply(lambda red: mode(red))
if 'Object' in full.columns:
    all_groups['Object'] = full_grouped['Object'].apply(lambda obj: mode(obj))

all_groups = all_groups.drop(
    columns=['SNR', 'Date', 'RV', 'Checked', 'Coordtype', 'Epoch', 'Epochsystem', 'Equinox', 'Parallax', 'PM Alpha', 'PM Delta', 'Airmass', 'RA', 'Dec'],
    errors='ignore'
)
all_groups = all_groups.reset_index(drop=False)

print(full)
print(all_groups)
full.to_pickle("/home/msp25gd/Downloads/res/meta/full_metadata.pkl")
all_groups.to_pickle("/home/msp25gd/Downloads/res/meta/groups.pkl")

metadata + spectra loaded
                  OBJECT          Sanitised         Reduced          RA  \
1           HE 0048-6408       HE 0048 6408      he00486408   12.688136   
3         QSO B2139-4433     QSO B2139 4433    qsob21394433  325.592422   
5         QSO B2139-4433     QSO B2139 4433    qsob21394433  325.592345   
7      BPS CS 30324-0063  BPS CS 30324 0063  bpscs303240063    5.686189   
9            J22564-5910        J22564-5910     j22564-5910  344.102157   
...                  ...                ...             ...         ...   
84169           HD 48279           HD 48279         hd48279  100.669108   
84171          HD 58465A          HD 58465A        hd58465a  111.254815   
84181           HD 42379           HD 42379         hd42379   92.825708   
84182           HD 42088           HD 42088         hd42088   92.415241   
84183           HD 39746           HD 39746         hd39746   88.918130   

            DEC    EXPTIME       MJD-OBS       MJD-END    WAVELMIN  \
1  

In [5]:
print(all_groups)

       New Groups                   OBJECT                Sanitised  \
0               0             $\alpha$-Cru             $\alpha$ Cru   
1               1           $\gamma^2$-Vel           $\gamma^2$ Vel   
2               2             $\theta$-Car             $\theta$ Car   
3               3              $\zeta$-Pup              $\zeta$ Pup   
4               4                0003+1713                0003+1713   
...           ...                      ...                      ...   
12822       12822                  zetaPup                  zetaPup   
12823       12823                 zeta-Tau                 zeta Tau   
12824       12824                  zet Per                  zet Per   
12825       12825                    Z-Sct                    Z Sct   
12826       12826  ZTF_J175542.24+055209.8  ZTF J175542.24+055209.8   

                      Reduced       DEC    EXPTIME       MJD-OBS  \
0                 $\alpha$cru -63.09899     5.0078  53747.372964   
1          